# gregorithm quickstart

This notebook shows the normal workflow. `gregorithm` consumes standardized data with
`time_s`, `current_mA` and `voltage_V`. Use `battread` first for vendor exports or other
column names. Change only `DATA` and `WORK` first. Results are ordinary tables or
interactive figures; large point data remain streamed.

In [ ]:
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from gregorithm import PulseProfile, open_result, prepare_source

WORK = Path("analysis-output")
WORK.mkdir(exist_ok=True)

For example, standardize a raw export once with the separate `battread`
library, then point `DATA` to the resulting Parquet file.

In [ ]:
# import battread
# battread.convert("raw-export.csv", "cell.standardized.parquet", chunk_size=10_000)
# DATA = Path("cell.standardized.parquet")

The quickstart is self-contained by default. This cell creates a small,
clearly synthetic charge/discharge and pulse trace. Replace `DATA` with your standardized
file path after the first run.

In [ ]:
DATA = WORK / "synthetic-demo.parquet"

parts = [
    (np.arange(0, 11), 0.0, np.linspace(3.0, 3.0, 11)),
    (np.arange(11, 112), 10.0, np.linspace(3.0, 4.0, 101)),
    (np.arange(112, 123), 0.0, np.linspace(4.0, 4.0, 11)),
    (np.arange(123, 224), -10.0, np.linspace(4.0, 3.0, 101)),
    (np.arange(224, 235), 0.0, np.linspace(3.0, 3.0, 11)),
    (np.arange(235, 266), -20.0, np.linspace(2.98, 2.94, 31)),
    (np.arange(266, 277), 0.0, np.linspace(2.96, 2.97, 11)),
]
time_s = np.concatenate([part[0] for part in parts]).astype(float)
current_mA = np.concatenate([
    np.full(len(part[0]), part[1], dtype=float) for part in parts
])
voltage_V = np.concatenate([part[2] for part in parts]).astype(float)
pq.write_table(
    pa.table({"time_s": time_s, "current_mA": current_mA, "voltage_V": voltage_V}),
    DATA,
)

## 1. Prepare and inspect

Preparation preserves source columns. Duplicate times keep the first complete row and
warn; backwards distinct times raise an error.

In [ ]:
cell = prepare_source(DATA, WORK / "cell.prepared.parquet")
cell.preview(5)

## 2. Capacity, energy and cycle summaries

In [ ]:
capacity = cell.capacity(include_energy=True)
display(capacity.table())
display(capacity.summary().table())
capacity.plot(y="capacity_mAh").figure

Normalize only when you have measured mass or area. Both may be supplied;
existing result columns are preserved.

In [ ]:
normalized = capacity.normalize(mass_g=0.005, area_cm2=1.13)
normalized.table()

## 3. C-rate, cumulative values and an exact time window

In [ ]:
display(cell.c_rate(capacity=capacity).preview())
display(cell.cumulative(include_energy=True).preview())
cell.time_window(120, 180, include_energy=True).table()

## 4. Optional step classification

Step labels are calculated only here. `measurements()` is the explicit request for a
point table containing `step_type`.

In [ ]:
steps = cell.steps()
display(steps.table())
display(steps.quantities(include_energy=True).table())
steps.measurements().preview()

## 5. Pulse discovery and DCIR

Discovery returns candidates. DCIR is a separate calculation at one requested delay.
Unavailable measurements remain visible with a reason.

In [ ]:
events = cell.events()
display(events.table())
events.dcir(delay_s=10).table()

If the protocol durations are known, check candidates against them. Set a
missing pre- or post-rest duration to zero.

In [ ]:
profile = PulseProfile(
    rest_before_s=10,
    pulse_s=30,
    rest_after_s=10,
    rest_before_tolerance_s=2,
    pulse_tolerance_s=2,
    rest_after_tolerance_s=2,
)
cell.events(profile=profile).table()

## 6. dQ/dV

Choose the cycles to inspect. The default is the corrected historical method and uses
continuous CC support only; CV portions are excluded.

In [ ]:
curves = cell.dqdv(cycles=[1])
display(curves.outcomes())
curves.plot().figure

## 7. Original signals and progressive plotting

In [ ]:
cell.plot(y="voltage_V").figure

# In an interactive kernel:
# session = cell.plot_progressive(y="voltage_V")
# display(session.figure)
# session.start()

## 8. Save and reopen a result

In [ ]:
capacity.write_parquet(WORK / "capacity.parquet")
saved = open_result(WORK / "capacity.parquet")
saved.table()

For large point-level results, use `iter_batches()` or `write_parquet()`.
Specialist GITT and ICI calculations are outside this release.